# Build a Text Generator

**Goal:** Build a neural network model that generates text sequences step by step.


Text generation models learn language patterns from training data.  
They predict the next word given a sequence of previous words, and repeat this process to form sentences.

---

## Steps
1. Prepare text corpus.
2. Tokenize and create sequences.
3. Build and train a model.
4. Generate text by predicting next words.

---

## Layman Explanation
Think of this as teaching a **robot storyteller**:
- Step 1: Give it a library of books (training data).
- Step 2: Teach it word order (tokenization).
- Step 3: Build its brain (model).
- Step 4: Ask it to continue a sentence, word by word.


In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

# Sample corpus
data = ["I love machine learning", 
        "Machine learning is fun", 
        "Deep learning powers AI", 
        "AI will change the world"]

# Tokenize
tokenizer = Tokenizer()
tokenizer.fit_on_texts(data)
total_words = len(tokenizer.word_index) + 1

# Create sequences
input_sequences = []
for line in data:
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_seq = token_list[:i+1]
        input_sequences.append(n_gram_seq)

# Pad sequences
max_seq_len = max([len(x) for x in input_sequences])
input_sequences = pad_sequences(input_sequences, maxlen=max_seq_len, padding='pre')

# Split predictors and label
X, y = input_sequences[:,:-1], input_sequences[:,-1]
y = tf.keras.utils.to_categorical(y, num_classes=total_words)

# Build model
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(total_words, 10, input_length=max_seq_len-1),
    tf.keras.layers.LSTM(100),
    tf.keras.layers.Dense(total_words, activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.fit(X, y, epochs=200, verbose=0)


C:\Users\singh\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\keras\src\layers\core\embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [2]:
def generate_text(seed_text, next_words, model, max_seq_len):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_seq_len-1, padding='pre')
        predicted = np.argmax(model.predict(token_list, verbose=0))
        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted:
                output_word = word
                break
        seed_text += " " + output_word
    return seed_text

print(generate_text("Machine learning", 5, model, max_seq_len))


Machine learning is fun fun world the


- We trained a model on a small text corpus.
- The model predicts the next word based on previous words.
- We used a function to generate sentences step by step.

Layman:
It’s like asking a robot storyteller:
- Start with a phrase.
- Robot guesses the next word.
- Keep asking until a full sentence is formed.
